# br_ana — local verification notebook

Covers both products from ANA Hidroweb:

| Product | Endpoint | Unit |
|---|---|---|
| `discharge_daily_mean` | HidroSerieVazao/v1 | m³/s (native, no conversion) |
| `stage_daily_mean` | HidroSerieCotas/v1 | cm → m (÷100) |

**Authentication required**: set `ANA_IDENTIFICADOR` and `ANA_SENHA` env vars before running observation cells.

Catalogue cells (sections 1–3) work offline — no credentials needed.

In [ ]:
import os
import polars as pl
import rivretrieve as rr
from datetime import datetime, UTC

print('RivRetrieve version:', rr.__version__)
print('Providers:', rr.providers())
assert 'br_ana' in rr.providers()
print('Has ANA credentials:', bool(os.environ.get('ANA_IDENTIFICADOR')))

## 1. Provider info

In [ ]:
provider = rr.provider('br_ana')
info = provider.info()
print('Provider ID:       ', info.provider_id)
print('Name:              ', info.name)
print('Catalogue version: ', info.catalogue_version)
info

## 2. Catalogue — stations

In [ ]:
import json

stations = provider.stations().data
print(f'Total stations: {stations.height}')
print('Countries:', stations['country'].unique().to_list())
print('Elevation null count:     ', stations['elevation_m'].null_count())
print('Drainage area null count: ', stations['drainage_area_km2'].null_count())
stations

## 3. Catalogue — products & station-products

In [ ]:
products = provider.products().data
print(f'Products: {products.height}')
products.select(['product_id', 'observed_property', 'frequency', 'statistic', 'unit'])

In [ ]:
sp = provider.station_products().data
print(f'Station-products total: {sp.height}')
sp.group_by('product_id').len().sort('product_id')

## 4. discharge_daily_mean — live download

Station **12345000** (Rio Xingu at Altamira, Pará). Requires ANA credentials.

If credentials are absent, the result will be empty with an `auth_missing` issue.

In [ ]:
STATION = '12345000'
START = datetime(2020, 1, 1, tzinfo=UTC)
END   = datetime(2020, 3, 31, tzinfo=UTC)

r_q = provider.observations(
    stations=STATION,
    products='discharge_daily_mean',
    start=START,
    end=END,
    on_issue='warn',
)

print(f'Rows: {r_q.data.height}')
print('Issues:', [(i.code, i.severity) for i in r_q.issues])
r_q.data.head(10)

In [ ]:
# Schema and UTC check
assert r_q.data.schema['time'] == pl.Datetime(time_unit='us', time_zone='UTC'), 'time column must be UTC'
print('UTC schema OK')

if not r_q.data.is_empty():
    times = r_q.data.sort('time')['time'].to_list()
    assert all(t.hour == 0 and t.minute == 0 for t in times[:10]), 'Timestamps must be UTC midnight'
    print('Midnight timestamps OK')
    vals = r_q.data['value'].drop_nulls().to_list()
    print(f'Value range: {min(vals):.1f}–{max(vals):.1f} m³/s')

    # Series annotations
    ann = r_q.series_annotations.data
    ann_dict = dict(zip(ann['annotation'].to_list(), ann['value'].to_list()))
    assert ann_dict['timezone_source'] == 'date_only_utc_midnight'
    assert ann_dict['date_only_timestamp_flag'] == 'true'
    assert ann_dict['resolved_timezone'] == 'UTC'
    assert ann_dict['native_unit_returned'] == 'm3/s'
    assert ann_dict['converted_unit'] == 'm3/s'
    print('Series annotation checks passed.')

    # Row annotations: no conversion (m³/s direct)
    row_ann = r_q.row_annotations.data
    raw_rows = row_ann.filter(pl.col('annotation') == 'raw_value').sort('time')
    first_raw = float(raw_rows['value'][0])
    first_val = r_q.data.sort('time')['value'][0]
    print(f'raw_value: {first_raw:.2f} m³/s  ==  {first_val:.2f} m³/s (no conversion)')
    assert abs(first_raw - first_val) < 0.001
    print('No-conversion check passed.')
else:
    print('No data returned — likely missing credentials (auth_missing issue expected).')
    auth_codes = [i.code for i in r_q.issues]
    print('Issue codes:', auth_codes)

In [ ]:
if not r_q.data.is_empty():
    df = r_q.data.to_pandas().set_index('time')
    ax = df['value'].plot(figsize=(12, 3), title=f'discharge_daily_mean — {STATION} [m³/s]')
    ax.set_ylabel('m³/s')

## 5. stage_daily_mean — live download with cm→m conversion

Station **60435000** (Rio São Francisco at Três Marias, Minas Gerais).  
Native unit: cm → ÷100 → m. Raw cm preserved in `raw_value` row annotation.

In [ ]:
STATION_STAGE = '60435000'
START_S = datetime(2020, 1, 1, tzinfo=UTC)
END_S   = datetime(2020, 3, 31, tzinfo=UTC)

r_h = provider.observations(
    stations=STATION_STAGE,
    products='stage_daily_mean',
    start=START_S,
    end=END_S,
    on_issue='warn',
)

print(f'Rows: {r_h.data.height}')
print('Issues:', [(i.code, i.severity) for i in r_h.issues])
r_h.data.head(10)

In [ ]:
assert r_h.data.schema['time'] == pl.Datetime(time_unit='us', time_zone='UTC'), 'time column must be UTC'
print('UTC schema OK')

if not r_h.data.is_empty():
    vals = r_h.data['value'].drop_nulls().to_list()
    print(f'Value range: {min(vals):.2f}–{max(vals):.2f} m')
    assert all(v >= 0 for v in vals), 'Stage must be non-negative'

    ann = r_h.series_annotations.data
    ann_dict = dict(zip(ann['annotation'].to_list(), ann['value'].to_list()))
    assert ann_dict['timezone_source'] == 'date_only_utc_midnight'
    assert ann_dict['native_unit_returned'] == 'cm'
    assert ann_dict['converted_unit'] == 'm'
    print('Series annotation checks passed.')

    # cm → m: raw_value (cm) / 100 = value (m)
    row_ann = r_h.row_annotations.data
    raw_rows = row_ann.filter(pl.col('annotation') == 'raw_value').sort('time')
    first_raw = float(raw_rows['value'][0])
    first_val = r_h.data.sort('time')['value'][0]
    print(f'raw_value: {first_raw:.1f} cm  →  {first_raw/100:.3f} m  (reported: {first_val:.3f} m)')
    assert abs(first_raw / 100 - first_val) < 0.001
    print('cm ÷ 100 = m check passed.')
else:
    print('No data returned — likely missing credentials.')

In [ ]:
if not r_h.data.is_empty():
    df = r_h.data.to_pandas().set_index('time')
    ax = df['value'].plot(figsize=(12, 3), title=f'stage_daily_mean — {STATION_STAGE} [m]')
    ax.set_ylabel('m')

## 6. Auth-missing handling — no credentials

In [ ]:
from rivretrieve._internal.providers.br_ana.observation_client import BrAnaObservationClient
from rivretrieve._internal.providers.br_ana.retrieval import retrieve_observations
from rivretrieve._internal.observations import ObservationRequest

no_cred_factory = lambda: BrAnaObservationClient(username=None, password=None)

req = ObservationRequest(
    provider_id='br_ana',
    stations=['12345000'],
    products=['discharge_daily_mean'],
    start=datetime(2020, 1, 1, tzinfo=UTC),
    end=datetime(2020, 1, 31, tzinfo=UTC),
)
result_no_cred = retrieve_observations(req, on_issue='ignore', client_factory=no_cred_factory)
print('Data rows:', result_no_cred.data.height, '(expected 0)')
print('Issues:', [(i.code, i.severity) for i in result_no_cred.issues])
assert result_no_cred.data.is_empty()
assert any('auth_missing' in i.code for i in result_no_cred.issues)
print('auth_missing issue confirmed — graceful degradation OK.')

## 7. Bulk request — two stations

In [ ]:
r_bulk = provider.observations(
    stations=['12345000', '60435000'],
    products=['discharge_daily_mean', 'stage_daily_mean'],
    start=datetime(2020, 1, 1, tzinfo=UTC),
    end=datetime(2020, 1, 31, tzinfo=UTC),
    on_issue='warn',
)

print(f'Bulk result rows: {r_bulk.data.height}')
if not r_bulk.data.is_empty():
    print('Station × product breakdown:')
    print(r_bulk.data.group_by(['station_id', 'product_id']).len().sort(['station_id', 'product_id']))
else:
    print('No data — credentials required for live fetch.')
print('Issues:', [(i.code, i.severity) for i in r_bulk.issues])

## 8. Map

In [ ]:
# Quick scatter map with matplotlib (packaged catalogue, 2 fixture stations)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(
    stations['longitude'].to_list(),
    stations['latitude'].to_list(),
    s=10, alpha=0.8, color='steelblue'
)
for row in stations.iter_rows(named=True):
    ax.annotate(row['station_id'], (row['longitude'], row['latitude']), fontsize=8)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'br_ana fixture stations ({stations.height} total in packaged catalogue)')
plt.tight_layout()

In [ ]:
# Full rr.map_stations() — needs live catalogue to be useful
m = rr.map_stations(provider='br_ana')
m

## 9. Summary

In [ ]:
print('=== br_ana verification summary ===')
print(f'Provider:           {info.provider_id}')
print(f'Catalogue version:  {info.catalogue_version}')
print(f'Stations (fixture): {stations.height}')
print(f'Products:           {products.height}')
print(f'Station-products:   {sp.height}')
print()
print('Product summary:')
for row in products.sort('product_id').iter_rows(named=True):
    meta = json.loads(row['metadata'])
    print(f"  {row['product_id']:30s}  unit={row['unit']}  native={meta['native_unit']}")
print()
print('NOTE: Observations require ANA_IDENTIFICADOR + ANA_SENHA env vars.')
print('NOTE: Packaged catalogue has fixture stations only.')
print('      Run generate_catalogue.py --live to build the full catalogue.')